In [ ]:
import os
import numpy as np
from shapely.wkt import loads
import bo_sheet.channel_preprocess_utility as cu
from bo_sheet.sim_wrapper_BO import run_crawling_sim, _create_simulation_movie, _calculate_robot_speed_center_point

fixed_parameters = {}

# Using the same PDMS-2 as in 13_7_2025_speed_measurement.xlsx
material_name = "PDMS_2"

fixed_parameters['robot_length'] = 8.9 # [mm]
fixed_parameters['robot_width'] = 3.0 # [mm]
fixed_parameters['robot_thickness'] = 0.18 # [mm]
fixed_parameters['robot_n_waveform'] = 1.0 # Number of the waveform of the sinusoidal magnetization

fixed_parameters['robot_density'] = 1.795 * 1e-3 # [g/mm^3]
fixed_parameters['robot_modulus'] = 1.961476719 # [MPa]
fixed_parameters['robot_magnetization_density'] = 166849.413596388 # [A/m]
target_dl = 0.4

# Applicable to all channel types =============================================================================

'simple_straight'
'arc'
'straight'
's_shaped'
'sinusoidal'
'arbitrary'

channel_type = cu.ChannelType.simple_straight.value

# You can load pre-generated arbitrary channel
use_preGenerated_channel = False
channel_directory = os.path.join(os.path.expanduser('~'),
                                      'Documents', 'GitHub', 'Bayesian_Optimization_for_Sheet_Robots',
                                      'tasks', '1_run_sim_trials')

total_channel_length = 40.0 # For arbitrary, recommend 200 or larger. [mm]

# Arbitrary is always non-constant width
constant_width = True

# If constant_width is True
width = 2.2 # [mm]

# If constant_width is False
num_width_ctrl = 4 # Must be >= 2
min_width = 1.0
max_width = 3.0
max_rate_width = 2.0

origin_offset = 0.0
N_points = 1000

# Applicable to arc, s_shaped, sinusoidal, arbitrary settings ================================================

min_curvature = 0.002
max_curvature = 0.02
max_rate_curvature = 0.001

# For s_shaped and sinusoidal
amplitude = 2.0

# For sinusoidal
frequency = 1.5

# For arc
radius = 10.0

# For arbitrary
n_bending = 4

fixed_parameters['backward'] = True

B_field = 0.05 # [T]
B_frequency = 4.0 # [Hz]

variation_level = 0.0

static_mu_wall = 14.999999999968873
kinetic_mu_wall = 7.059512526167598
static_mu_wall_end = 3.6633357913913183
kinetic_mu_wall_end = 3.663335790996258
static_mu_substrate = 0.0
kinetic_mu_substrate = 0.0
k_mag_den = 1.9154742669976665
k_modulus = 1.0

final_time = 3e3 # [ms]

fixed_parameters['robot_modulus'] = fixed_parameters['robot_modulus'] * k_modulus # [MPa]
fixed_parameters['robot_magnetization_density'] = fixed_parameters['robot_magnetization_density'] * k_mag_den

n_elem = int(fixed_parameters['robot_length'] / target_dl)
if n_elem * target_dl < fixed_parameters['robot_length']:
    n_elem += 1
else:
    n_elem = n_elem
dl = fixed_parameters['robot_length'] / n_elem  # [mm]
dt = dl / 100.0
fixed_parameters['robot_n_elem'] = n_elem

if use_preGenerated_channel != True:
    x_midline, y_midline, channel_polygon, left_bank, right_bank = cu.generate_closed_channel(
        total_channel_length, n_bending, num_width_ctrl, max_curvature, min_curvature, max_width,
        min_width, max_rate_curvature, max_rate_width,
        channel_type, width, amplitude, frequency, radius, constant_width,
        origin_offset, N_points, fixed_parameters['backward']
    )
    
    with open(os.path.join(channel_directory, channel_type, "polygon.wkt"), "w") as file:
        file.write(channel_polygon.wkt)
    file.close()

    channel_info = {
        "x_midline": x_midline,
        "y_midline": y_midline,
        "left_bank": left_bank,
        "right_bank": right_bank
    }

    np.savez(os.path.join(channel_directory, channel_type, 'channel_info'), **channel_info)
else:
    with open(os.path.join(channel_directory, channel_type, "polygon.wkt"), "r") as file:
        channel_polygon = loads(file.read())
        file.close()

    with np.load(os.path.join(channel_directory, channel_type, 'channel_info.npz')) as channel_info:
        x_midline = channel_info['x_midline']
        y_midline = channel_info['y_midline']
        left_bank = channel_info['left_bank']
        right_bank = channel_info['right_bank']
        channel_info.close()

fixed_parameters['channel_type'] = channel_type
fixed_parameters['dist_threshold'] = fixed_parameters['robot_thickness'] / 2.0

channel_seg_midline = np.column_stack((x_midline, y_midline))

robot_init_pos = cu.generate_fiber_in_segment(channel_seg_midline,
                                              L_fiber=fixed_parameters['robot_length'],
                                              offset_factor=0.02,
                                              N_fiber=fixed_parameters['robot_n_elem'])

robot_init_director = cu.compute_directors_from_positions(robot_init_pos)
robot_init_origin = robot_init_pos[:, 0]

fixed_parameters['channel_polygon'] = channel_polygon
fixed_parameters['channel_x_mid'] = x_midline
fixed_parameters['channel_y_mid'] = y_midline
fixed_parameters['robot_init_pos'] = robot_init_pos
fixed_parameters['robot_init_director'] = robot_init_director
fixed_parameters['robot_init_origin'] = robot_init_origin

fixed_parameters['B_field'] = B_field
fixed_parameters['B_frequency'] = B_frequency

kinetic_mu_substrate_array = np.full((n_elem + 1,), kinetic_mu_substrate)
static_mu_substrate_array = np.full((n_elem + 1,), static_mu_substrate)
kinetic_mu_wall_array = np.full((n_elem + 1,), kinetic_mu_wall)
static_mu_wall_array = np.full((n_elem + 1,), static_mu_wall)

kinetic_mu_wall_array[:1] = kinetic_mu_wall_end
kinetic_mu_wall_array[-1:] = kinetic_mu_wall_end
static_mu_wall_array[:1] = static_mu_wall_end
static_mu_wall_array[-1:] = static_mu_wall_end

fixed_parameters["kinetic_mu_substrate_array"] = kinetic_mu_substrate_array
fixed_parameters["static_mu_substrate_array"] = static_mu_substrate_array
fixed_parameters["kinetic_mu_wall_array"] = kinetic_mu_wall_array
fixed_parameters["static_mu_wall_array"] = static_mu_wall_array
fixed_parameters['dt'] = dt
fixed_parameters['final_time'] = final_time
fixed_parameters['variation_level'] = variation_level

n_elem = robot_init_pos.shape[1] - 1
magnetization_direction = np.zeros((3, n_elem))

magnetization_angles = np.linspace(0, 2 * np.pi * fixed_parameters['robot_n_waveform'], n_elem)
magnetization_direction[0, :] = np.cos(magnetization_angles)
magnetization_direction[1, :] = np.sin(magnetization_angles)

cu.plot_channel_and_fiber(
    fixed_parameters['channel_polygon'],
    channel_seg_midline,
    fixed_parameters['robot_init_pos'],
    "Verify Desired Channel and Robot Setup"
)

post_processing_dict, actual_final_time_in_ms = run_crawling_sim(
    fixed_parameters['robot_length'],
    fixed_parameters['robot_width'],
    fixed_parameters['robot_thickness'],
    fixed_parameters['robot_n_waveform'],
    fixed_parameters['robot_density'],
    fixed_parameters['robot_modulus'],
    fixed_parameters['robot_magnetization_density'],
    fixed_parameters['channel_polygon'],
    fixed_parameters['dist_threshold'],
    fixed_parameters['robot_init_pos'],
    fixed_parameters['robot_init_origin'],
    fixed_parameters['robot_init_director'],
    fixed_parameters['B_field'],
    fixed_parameters['B_frequency'],
    fixed_parameters['kinetic_mu_substrate_array'],
    fixed_parameters['static_mu_substrate_array'],
    fixed_parameters['kinetic_mu_wall_array'],
    fixed_parameters['static_mu_wall_array'],
    fixed_parameters['dt'],
    fixed_parameters['final_time'],
    magnetization_direction,
    fixed_parameters['variation_level'],
    backward=fixed_parameters['backward'],
    n_elem=n_elem
)

positions_over_time = np.array(post_processing_dict["position"])

speed = _calculate_robot_speed_center_point(positions_over_time, actual_final_time_in_ms, x_midline, y_midline)

print(f"Speed {speed:.3f}mm/s")

_create_simulation_movie(
    positions_over_time,
    fixed_parameters['channel_polygon'],
    f"stat_mu_sub_{static_mu_substrate:.1f}_kine_mu_sub_{kinetic_mu_substrate}_k_modulus_{k_modulus:.1f}_channel_width_{width:.1f}mm_speed_{speed:.3f}mms.mp4",
    os.path.join(channel_directory, fixed_parameters['channel_type'])
)